# Task 1 — Object Detection with a CNN Architecture (YOLOv11)

**Goal:** build an end-to-end object-detection system with **Ultralytics YOLOv11** that classifies *and* localises multiple objects per image.

| Step | What this notebook does |
|---|---|
| 1 | **Pre-process** – audit/clean the YOLO-format annotations, letterbox-resize to 640×640, normalise pixels to [0,1], inspect boxes |
| 2 | **Implement YOLOv11** – build `yolo11s`, inspect backbone / neck / head, parameters, GFLOPs |
| 3 | **Train** – fine-tune COCO-pre-trained weights on a *custom* dataset (African wildlife: buffalo, elephant, rhino, zebra) on the Kaggle GPU |
| 4 | **Evaluate** – mAP@0.5, mAP@0.5:0.95, precision/recall, per-class AP, IoU analysis, inference speed |
| 5 | **Unseen data** – held-out test split, qualitative grids, failure cases, robustness to corruptions |
| 6 | **Live prediction on frames** – frame-by-frame video loop with FPS overlay + multi-object tracking, exported to MP4/GIF |
| 7 | **Deliverables** – `best.pt`, ONNX export, metrics JSON, plots, videos |

> **Runs on Kaggle *or* Google Colab (GPU).** Colab: `Runtime > Change runtime type > T4 GPU`, then `Runtime > Run all` – the dataset is downloaded automatically (public Kaggle dataset). Outputs go to `/kaggle/working` or `/content`; a `yolo11_deliverables.zip` is produced (auto-downloaded on Colab).

In [ ]:
import os, sys, subprocess, json, time, random, shutil, glob, math, warnings, platform
warnings.filterwarnings("ignore")
os.environ["WANDB_MODE"] = "disabled"; os.environ["COMET_MODE"] = "disabled"

# SMOKE=1 -> tiny debug run (1 epoch, 5% of train set).  Final run uses SMOKE=0.
SMOKE = os.environ.get("YOLO_SMOKE", "0") == "1"

r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "ultralytics"], capture_output=True, text=True)
print(r.stdout[-500:], r.stderr[-500:])

import numpy as np, pandas as pd, cv2, torch, yaml
import matplotlib
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image
from IPython.display import display, Image as IPyImage
import ultralytics
from ultralytics import YOLO, settings
for _k in ("wandb", "comet", "mlflow", "tensorboard", "clearml", "neptune"):
    try: settings.update({_k: False})
    except Exception: pass

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

WORK = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("/content")   # Kaggle or Colab
WORK.mkdir(parents=True, exist_ok=True)
OUT = WORK / "outputs"; FIG = OUT / "figures"; FIG.mkdir(parents=True, exist_ok=True)
RUNS = WORK / "runs"

print("Python", platform.python_version(), "| torch", torch.__version__, "| ultralytics", ultralytics.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i); print(f"  GPU{i}: {p.name}, {p.total_memory/1e9:.1f} GB, sm_{p.major}{p.minor}")
if not torch.cuda.is_available(): print("WARNING: no GPU found - on Colab use Runtime > Change runtime type > T4 GPU; on Kaggle enable Accelerator = GPU.")
DEVICE = 0 if torch.cuda.is_available() else "cpu"
GPU_NAME = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
print("Using device:", DEVICE, "| SMOKE mode:", SMOKE)

## 1. Data — loading & pre-processing

**Dataset:** *Object Detection – Wildlife Dataset – YOLO Format* (Kaggle, `ankanghosh651`). It is a custom 4-class dataset (buffalo, elephant, rhino, zebra) already split into `train / valid / test`; annotations are YOLO text files `class x_center y_center width height` (normalised 0-1).
The test split is **never** used for training or model selection, so it is our *unseen* data.

In [ ]:
DATASET_ID = "ankanghosh651/object-detection-wildlife-dataset-yolo-format"
def find_dataset_yaml():
    if Path("/kaggle/input").exists():                                   # Kaggle: dataset attached to the notebook
        hits = list(Path("/kaggle/input").rglob("data_wl.yaml"))
        if hits: return hits[0]
    dl = WORK / "dl"; hits = list(dl.rglob("data_wl.yaml")) if dl.exists() else []
    if hits: return hits[0]
    try:                                                                  # Colab / local: public dataset via kagglehub
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kagglehub"], check=True)
        import kagglehub; hits = list(Path(kagglehub.dataset_download(DATASET_ID)).rglob("data_wl.yaml"))
        if hits: return hits[0]
    except Exception as e: print("kagglehub download failed:", repr(e)[:200])
    import getpass                                                        # fallback: Kaggle API token (never stored in the notebook)
    os.environ["KAGGLE_API_TOKEN"] = getpass.getpass("Kaggle API token (kaggle.com > Settings > API): ")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kaggle"], check=True); dl.mkdir(parents=True, exist_ok=True)
    subprocess.run(["kaggle", "datasets", "download", "-d", DATASET_ID, "-p", str(dl), "--unzip"], check=True)
    return next(dl.rglob("data_wl.yaml"))
SRC_YAML = find_dataset_yaml()
SRC = SRC_YAML.parent
print("Dataset found at:", SRC)
DATA = WORK / "wildlife"
if DATA.exists(): shutil.rmtree(DATA)
shutil.copytree(SRC, DATA, ignore=shutil.ignore_patterns("*.cache", "*.yaml"))   # copy to writable disk
for c in DATA.rglob("*.cache"): c.unlink()

names_src = yaml.safe_load(open(SRC_YAML))["names"]
NAMES = [names_src[i] for i in sorted(names_src)] if isinstance(names_src, dict) else list(names_src)
NC = len(NAMES)

split_dirs = {s: DATA / d for s, d in {"train": "train", "val": "valid", "test": "test"}.items()}
for s, d in split_dirs.items():
    assert (d / "images").exists(), f"missing {d/'images'}"
    print(f"{s:5s} -> {d.name:6s}: {len(list((d/'images').glob('*')))} images, {len(list((d/'labels').glob('*.txt')))} label files")

DATA_YAML = DATA / "wildlife.yaml"
yaml.safe_dump({"path": str(DATA), "train": "train/images", "val": "valid/images", "test": "test/images",
                "nc": NC, "names": {i: n for i, n in enumerate(NAMES)}}, open(DATA_YAML, "w"))
print(open(DATA_YAML).read())

In [ ]:
# ---- Annotation audit: format validity, class balance, box statistics, image sizes ----
IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
def list_images(split): return sorted(p for p in (split_dirs[split]/"images").iterdir() if p.suffix.lower() in IMG_EXT)

def parse_label_file(lp):
    # returns array [n,5] of (cls, xc, yc, w, h) normalised; handles polygon lines by taking their bbox; counts problems
    rows, bad = [], 0
    if not lp.exists(): return np.zeros((0,5)), 0
    for line in lp.read_text().strip().splitlines():
        p = line.split()
        try:
            if len(p) == 5:
                c, xc, yc, w, h = int(float(p[0])), *map(float, p[1:])
            elif len(p) > 5 and len(p) % 2 == 1:           # polygon -> bounding box
                xs, ys = np.array(p[1::2], float), np.array(p[2::2], float)
                c = int(float(p[0])); xc, yc = (xs.min()+xs.max())/2, (ys.min()+ys.max())/2; w, h = xs.max()-xs.min(), ys.max()-ys.min()
            else: bad += 1; continue
        except ValueError: bad += 1; continue
        if not (0 <= c < NC) or not (0 <= xc <= 1 and 0 <= yc <= 1 and 0 < w <= 1.0001 and 0 < h <= 1.0001): bad += 1; continue
        rows.append((c, xc, yc, w, h))
    return np.array(rows).reshape(-1, 5), bad

audit, boxes_by_split = [], {}
for s in ["train", "val", "test"]:
    imgs = list_images(s); allb, bad, bg = [], 0, 0
    for ip in imgs:
        b, nb_ = parse_label_file(split_dirs[s]/"labels"/f"{ip.stem}.txt"); bad += nb_; bg += int(len(b) == 0); allb.append(b)
    B = np.concatenate(allb) if allb else np.zeros((0,5)); boxes_by_split[s] = B
    sizes = [Image.open(p).size for p in random.Random(0).sample(imgs, min(150, len(imgs)))]
    audit.append({"split": s, "images": len(imgs), "boxes": len(B), "boxes/img": round(len(B)/max(1,len(imgs)),2),
                  "background_imgs": bg, "invalid_lines": bad, "median_WxH": f"{int(np.median([a for a,_ in sizes]))}x{int(np.median([b for _,b in sizes]))}",
                  **{f"n_{n}": int((B[:,0]==i).sum()) for i, n in enumerate(NAMES)}})
audit_df = pd.DataFrame(audit); display(audit_df); audit_df.to_csv(OUT/"dataset_audit.csv", index=False)

fig, ax = plt.subplots(1, 3, figsize=(17, 4))
w = 0.27
for k, s in enumerate(["train", "val", "test"]):
    cnt = np.bincount(boxes_by_split[s][:,0].astype(int), minlength=NC); ax[0].bar(np.arange(NC)+(k-1)*w, cnt, w, label=s)
ax[0].set_xticks(range(NC)); ax[0].set_xticklabels(NAMES); ax[0].set_title("Boxes per class"); ax[0].legend()
Bt = boxes_by_split["train"]
ax[1].scatter(Bt[:,3], Bt[:,4], s=4, alpha=.35, c=Bt[:,0], cmap="tab10"); ax[1].set_xlabel("box width (norm)"); ax[1].set_ylabel("box height (norm)"); ax[1].set_title("Box size distribution (train)")
ax[2].hist(Bt[:,3]*Bt[:,4], bins=40, color="tab:purple"); ax[2].set_xlabel("box area / image area"); ax[2].set_title("Relative box area (train)")
plt.tight_layout(); plt.savefig(FIG/"01_dataset_stats.png", dpi=130); plt.show()

In [ ]:
# ---- Annotated samples ----
COLORS = [(230,57,70),(29,161,242),(46,204,113),(241,196,15),(155,89,182),(255,127,14)]
def draw_gt(img_bgr, lab):
    h, w = img_bgr.shape[:2]; im = img_bgr.copy()
    for c, xc, yc, bw, bh in lab:
        x1, y1, x2, y2 = int((xc-bw/2)*w), int((yc-bh/2)*h), int((xc+bw/2)*w), int((yc+bh/2)*h)
        col = COLORS[int(c) % len(COLORS)][::-1]; cv2.rectangle(im, (x1,y1), (x2,y2), col, max(2, w//300))
        cv2.putText(im, NAMES[int(c)], (x1, max(15, y1-5)), cv2.FONT_HERSHEY_SIMPLEX, max(.6, w/1000), col, 2)
    return im

train_imgs = list_images("train"); rng = random.Random(1)
sample = rng.sample(train_imgs, 8)
fig, axs = plt.subplots(2, 4, figsize=(18, 8))
for a, p in zip(axs.ravel(), sample):
    lab, _ = parse_label_file(split_dirs["train"]/"labels"/f"{p.stem}.txt"); a.imshow(draw_gt(cv2.imread(str(p)), lab)[..., ::-1]); a.set_title(f"{p.name}  ({len(lab)} boxes)", fontsize=9); a.axis("off")
plt.suptitle("Ground-truth annotations (train)"); plt.tight_layout(); plt.savefig(FIG/"02_gt_samples.png", dpi=110); plt.show()

In [ ]:
# ---- Pre-processing demo: letterbox resize -> 640x640, BGR->RGB, HWC->CHW, uint8 -> float32 in [0,1] ----
from ultralytics.data.augment import LetterBox
IMGSZ = 640
p = sample[0]; img = cv2.imread(str(p)); h0, w0 = img.shape[:2]
lab, _ = parse_label_file(split_dirs["train"]/"labels"/f"{p.stem}.txt")
lb = LetterBox(new_shape=(IMGSZ, IMGSZ), auto=False, scaleup=True, center=True)
img_lb = lb(image=img)
r_ = min(IMGSZ/h0, IMGSZ/w0); nh, nw = round(h0*r_), round(w0*r_); padw, padh = (IMGSZ-nw)/2, (IMGSZ-nh)/2
lab_lb = lab.copy()                         # transform labels to letterboxed normalised coords
lab_lb[:,1] = (lab[:,1]*nw + padw)/IMGSZ; lab_lb[:,2] = (lab[:,2]*nh + padh)/IMGSZ; lab_lb[:,3] = lab[:,3]*nw/IMGSZ; lab_lb[:,4] = lab[:,4]*nh/IMGSZ
tensor = torch.from_numpy(np.ascontiguousarray(img_lb[..., ::-1].transpose(2,0,1))).float()/255.0
print(f"original  : {w0}x{h0}  dtype={img.dtype}  range=[{img.min()}, {img.max()}]")
print(f"letterbox : {img_lb.shape[1]}x{img_lb.shape[0]}  (scale={r_:.3f}, pad=({padw:.0f},{padh:.0f}) grey 114)")
print(f"tensor    : {tuple(tensor.shape)}  dtype={tensor.dtype}  range=[{tensor.min():.2f}, {tensor.max():.2f}]  mean={tensor.mean():.3f}")
fig, ax = plt.subplots(1, 2, figsize=(14, 6))
ax[0].imshow(draw_gt(img, lab)[..., ::-1]); ax[0].set_title(f"Original {w0}x{h0}"); ax[0].axis("off")
ax[1].imshow(draw_gt(img_lb, lab_lb)[..., ::-1]); ax[1].set_title(f"Letterboxed {IMGSZ}x{IMGSZ} (boxes transformed)"); ax[1].axis("off")
plt.tight_layout(); plt.savefig(FIG/"03_preprocessing.png", dpi=110); plt.show()

**Pre-processing / augmentation pipeline used for training (handled by Ultralytics' data-loader):**
letterbox resize to 640×640 (aspect-ratio preserving, grey padding) → pixel normalisation `/255` → on-the-fly augmentation: *mosaic* (4-image tiles), random scale/translate, horizontal flip, HSV colour jitter, and *close-mosaic* (mosaic off for the last 10 epochs). Bounding boxes are transformed together with the pixels.

## 2. Implementing YOLOv11

YOLO11 is a single-stage, fully-convolutional, **anchor-free** detector. Its pipeline:

* **Backbone** – strided `Conv` (Conv-BN-SiLU) layers, **`C3k2`** blocks (CSP bottleneck with 2 small convs; replaces YOLOv8's C2f), **`SPPF`** (fast spatial pyramid pooling) and **`C2PSA`** (cross-stage partial block with Position-Sensitive self-Attention) at the deepest stage.
* **Neck** – PAN-FPN: top-down upsample + concat with backbone features, then bottom-up downsample + concat, producing three scales **P3/8, P4/16, P5/32** for small / medium / large objects.
* **Head** – decoupled **`Detect`** head per scale: one branch regresses box distribution (DFL, 4×16 bins), another classifies (depth-wise-separable convs in the cls branch in YOLO11). No anchors, no objectness score.
* **Loss** – CIoU box loss + Distribution-Focal Loss + BCE classification, with Task-Aligned Assigner for label assignment.

In [ ]:
model = YOLO("yolo11s.pt")           # COCO-pretrained weights (transfer learning)
n_layers, n_params, n_grad, gflops = model.info(detailed=False)
print("\nTask:", model.task, "| classes in pretrained head:", len(model.names))

In [ ]:
# ---- Layer-by-layer structure (backbone / neck / head) ----
seq = model.model.model
rows = []
for i, m in enumerate(seq):
    rows.append({"idx": i, "type": m.type.split(".")[-1], "from": m.f, "params": sum(p.numel() for p in m.parameters())})
layer_df = pd.DataFrame(rows)
layer_df["stage"] = np.where(layer_df.idx <= 10, "backbone", np.where(layer_df.idx < len(seq)-1, "neck", "head"))
display(layer_df)
stage_df = layer_df.groupby("stage", sort=False).params.sum().reset_index(); stage_df["share_%"] = (100*stage_df.params/stage_df.params.sum()).round(1); display(stage_df)
layer_df.to_csv(OUT/"yolo11s_layers.csv", index=False)

# model-family comparison built from the YAML configs (random init, no downloads)
fam = []
for v in ["n", "s", "m", "l", "x"]:
    m_ = YOLO(f"yolo11{v}.yaml"); nl, npar, _, gf = m_.info(verbose=False); fam.append({"variant": f"yolo11{v}", "layers": nl, "params_M": round(npar/1e6, 2), "GFLOPs@640": round(gf, 1)})
fam_df = pd.DataFrame(fam); display(fam_df); fam_df.to_csv(OUT/"yolo11_family.csv", index=False)

In [ ]:
# ---- Architecture diagram (generated from the real layer list) ----
import matplotlib.patches as mp
fig, ax = plt.subplots(figsize=(17, 6.2)); ax.set_xlim(0, 17); ax.set_ylim(0, 6.2); ax.axis("off")
def box(x, y, w, h, t, c, fs=8.5):
    ax.add_patch(mp.FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.04", fc=c, ec="#333", lw=1)); ax.text(x+w/2, y+h/2, t, ha="center", va="center", fontsize=fs)
def arr(x1, y1, x2, y2): ax.annotate("", xy=(x2, y2), xytext=(x1, y1), arrowprops=dict(arrowstyle="->", color="#444", lw=1.3))
bb = [("Input\n640x640x3", "#e8e8e8"), ("Conv 3x3/2\nP1/2", "#cfe8ff"), ("Conv /2 + C3k2\nP2/4", "#cfe8ff"), ("Conv /2 + C3k2\nP3/8", "#9fd0ff"), ("Conv /2 + C3k2\nP4/16", "#9fd0ff"), ("Conv /2 + C3k2\nP5/32", "#9fd0ff"), ("SPPF", "#ffd9a0"), ("C2PSA\n(attention)", "#ffd9a0")]
for k, (t, c) in enumerate(bb): box(0.2+k*1.35, 4.9, 1.2, 0.9, t, c, 7.5);  (arr(0.2+k*1.35+1.2, 5.35, 0.2+(k+1)*1.35, 5.35) if k < len(bb)-1 else None)
ax.text(5.3, 6.05, "BACKBONE", ha="center", fontweight="bold")
box(1.5, 2.6, 2.6, 0.8, "Upsample + Concat(P4) -> C3k2", "#d6f5d6"); box(4.6, 2.6, 2.6, 0.8, "Upsample + Concat(P3) -> C3k2  (P3 out)", "#d6f5d6")
box(7.7, 2.6, 2.6, 0.8, "Conv /2 + Concat -> C3k2  (P4 out)", "#d6f5d6"); box(10.8, 2.6, 2.6, 0.8, "Conv /2 + Concat -> C3k2  (P5 out)", "#d6f5d6")
for x in (4.1, 7.2, 10.3): arr(x, 3.0, x+0.5, 3.0)
ax.text(8.5, 3.55, "NECK  (PAN-FPN: top-down + bottom-up)", ha="center", fontweight="bold")
arr(9.3, 4.9, 9.3, 3.45)
for k, (lab, x) in enumerate([("P3/8 (small)", 6.0), ("P4/16 (medium)", 9.0), ("P5/32 (large)", 12.1)]):
    box(x-1.05, 0.8, 2.1, 0.9, f"Detect head\n{lab}", "#ffc2c2", 8.5); arr(x, 2.6, x, 1.72)
ax.text(8.5, 0.25, "HEAD: decoupled box branch (DFL, 4x16) + class branch (anchor-free)  ->  boxes, class scores  ->  NMS", ha="center", fontsize=9)
plt.title("YOLO11 architecture (yolo11s)", fontsize=12); plt.tight_layout(); plt.savefig(FIG/"04_architecture.png", dpi=130); plt.show()

## 3. Training

**Transfer learning:** the COCO-pre-trained `yolo11s.pt` is fine-tuned on the wildlife dataset (the 80-class head is re-initialised for our 4 classes). Settings: 640 px, batch 16, AdamW/SGD chosen by `optimizer="auto"`, cosine-free linear LR decay, AMP (FP16) on the Kaggle GPU, early stopping (`patience=10`), best checkpoint selected on the **validation** split.

In [ ]:
EPOCHS   = 1 if SMOKE else 30
BATCH    = 16
FRACTION = 0.05 if SMOKE else 1.0
RUN_NAME = "yolo11s_wildlife"
t0 = time.time()
results = model.train(data=str(DATA_YAML), epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, device=DEVICE, workers=2,
                      project=str(RUNS), name=RUN_NAME, exist_ok=True, seed=SEED, patience=10, optimizer="auto",
                      amp=True, plots=True, fraction=FRACTION, close_mosaic=10, pretrained=True, verbose=True)
TRAIN_MIN = (time.time()-t0)/60
SAVE = Path(results.save_dir); BEST = SAVE/"weights"/"best.pt"
print(f"\nTraining finished in {TRAIN_MIN:.1f} min  ->  {SAVE}\nbest weights: {BEST} ({BEST.stat().st_size/1e6:.1f} MB)")

In [ ]:
# ---- Training curves ----
hist = pd.read_csv(SAVE/"results.csv"); hist.columns = [c.strip() for c in hist.columns]; hist.to_csv(OUT/"training_history.csv", index=False)
fig, ax = plt.subplots(2, 4, figsize=(20, 8)); ax = ax.ravel()
for a, c in zip(ax[:6], ["train/box_loss","train/cls_loss","train/dfl_loss","val/box_loss","val/cls_loss","val/dfl_loss"]):
    a.plot(hist.epoch, hist[c], marker="o", ms=3); a.set_title(c); a.grid(alpha=.3)
ax[6].plot(hist.epoch, hist["metrics/precision(B)"], label="precision"); ax[6].plot(hist.epoch, hist["metrics/recall(B)"], label="recall"); ax[6].legend(); ax[6].set_title("val precision / recall"); ax[6].grid(alpha=.3)
ax[7].plot(hist.epoch, hist["metrics/mAP50(B)"], label="mAP@0.5"); ax[7].plot(hist.epoch, hist["metrics/mAP50-95(B)"], label="mAP@0.5:0.95"); ax[7].legend(); ax[7].set_title("val mAP"); ax[7].grid(alpha=.3)
for a in ax: a.set_xlabel("epoch")
plt.tight_layout(); plt.savefig(FIG/"05_training_curves.png", dpi=120); plt.show()
print(hist[["epoch","train/box_loss","val/box_loss","metrics/precision(B)","metrics/recall(B)","metrics/mAP50(B)","metrics/mAP50-95(B)"]].tail(5).round(4).to_string(index=False))

In [ ]:
# ---- Ultralytics-generated diagnostics ----
for f in ["labels.jpg", "train_batch0.jpg", "results.png", "confusion_matrix_normalized.png", "BoxPR_curve.png", "BoxF1_curve.png", "val_batch0_labels.jpg", "val_batch0_pred.jpg"]:
    fp = SAVE/f
    if fp.exists():
        plt.figure(figsize=(13, 7) if "batch" in f or f=="labels.jpg" else (11, 7)); plt.imshow(Image.open(fp)); plt.axis("off"); plt.title(f); plt.tight_layout(); plt.show()

## 4. Evaluation

Metrics (COCO-style, computed by Ultralytics' validator on the **validation** and the **held-out test** split, `conf=0.001`, NMS-IoU 0.6):
* **mAP@0.5** – mean AP with a detection counted correct if IoU ≥ 0.5.
* **mAP@0.5:0.95** – mean AP over IoU thresholds 0.50…0.95 (step 0.05) – the strict localisation-aware metric.
* **Precision / Recall**, **per-class AP**, **confusion matrix**.
* **IoU** – measured directly between predicted and ground-truth boxes (section 4.2).
* **Inference speed** – latency and FPS (section 4.3).

In [ ]:
best = YOLO(str(BEST))
def eval_split(split):
    m = best.val(data=str(DATA_YAML), split=split, imgsz=IMGSZ, batch=16, conf=0.001, iou=0.6, device=DEVICE, plots=True,
                 project=str(RUNS), name=f"eval_{split}", exist_ok=True, verbose=False)
    per = []
    for k, ci in enumerate(m.box.ap_class_index):
        p, r, ap50, ap = m.box.class_result(k); per.append({"class": m.names[int(ci)], "precision": p, "recall": r, "AP@0.5": ap50, "AP@0.5:0.95": ap})
    summ = {"split": split, "precision": float(m.box.mp), "recall": float(m.box.mr), "mAP@0.5": float(m.box.map50), "mAP@0.75": float(m.box.map75), "mAP@0.5:0.95": float(m.box.map),
            "F1": float(2*m.box.mp*m.box.mr/max(1e-9, m.box.mp+m.box.mr)), "speed_ms": {k: float(v) for k, v in m.speed.items()}}
    return m, summ, pd.DataFrame(per)

m_val, S_val, PC_val = eval_split("val")
m_test, S_test, PC_test = eval_split("test")
summary_df = pd.DataFrame([{k: v for k, v in S.items() if k != "speed_ms"} for S in (S_val, S_test)]).set_index("split")
print("=== Overall metrics ==="); display(summary_df.round(4))
print("\n=== Per-class (TEST split) ==="); display(PC_test.round(4))
print("\n=== Per-class (VAL split) ===");  display(PC_val.round(4))
summary_df.to_csv(OUT/"metrics_overall.csv"); PC_test.to_csv(OUT/"metrics_per_class_test.csv", index=False); PC_val.to_csv(OUT/"metrics_per_class_val.csv", index=False)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(15, 4.5))
x = np.arange(len(PC_test)); w = .38
ax[0].bar(x-w/2, PC_test["AP@0.5"], w, label="AP@0.5"); ax[0].bar(x+w/2, PC_test["AP@0.5:0.95"], w, label="AP@0.5:0.95"); ax[0].set_xticks(x); ax[0].set_xticklabels(PC_test["class"]); ax[0].set_ylim(0, 1.05); ax[0].legend(); ax[0].set_title("Per-class AP (test)")
for i, v in enumerate(PC_test["AP@0.5"]): ax[0].text(i-w/2, v+.01, f"{v:.3f}", ha="center", fontsize=8)
for i, v in enumerate(PC_test["AP@0.5:0.95"]): ax[0].text(i+w/2, v+.01, f"{v:.3f}", ha="center", fontsize=8)
ss = summary_df[["precision","recall","mAP@0.5","mAP@0.5:0.95"]].T; ss.plot.bar(ax=ax[1], rot=0); ax[1].set_ylim(0, 1.05); ax[1].set_title("Overall metrics: val vs test")
plt.tight_layout(); plt.savefig(FIG/"06_metrics_test.png", dpi=120); plt.show()
for f in ["confusion_matrix_normalized.png", "BoxPR_curve.png"]:
    fp = Path(m_test.save_dir)/f
    if fp.exists(): plt.figure(figsize=(10, 7)); plt.imshow(Image.open(fp)); plt.axis("off"); plt.title(f"TEST: {f}"); plt.tight_layout(); plt.savefig(FIG/f"07_test_{f}", dpi=110); plt.show()

### 4.2 IoU analysis (predicted box vs. ground-truth box)
Predictions (`conf ≥ 0.25`, NMS-IoU 0.7) are matched greedily, highest confidence first, to unmatched ground-truth boxes of the **same class**; a match counts as a true positive when **IoU ≥ 0.5**. We report TP/FP/FN, precision/recall/F1 at that operating point, the **mean IoU of true positives**, and the **mean best-IoU per ground-truth object** (0 when the object is missed).

In [ ]:
from torchvision.ops import box_iou
test_imgs = list_images("test")

def gt_boxes(ip, shape, label_dir=None):
    lab, _ = parse_label_file(Path(label_dir or split_dirs["test"]/"labels")/f"{ip.stem}.txt"); h, w = shape
    xyxy = np.stack([(lab[:,1]-lab[:,3]/2)*w, (lab[:,2]-lab[:,4]/2)*h, (lab[:,1]+lab[:,3]/2)*w, (lab[:,2]+lab[:,4]/2)*h], 1) if len(lab) else np.zeros((0,4))
    return xyxy, lab[:,0].astype(int) if len(lab) else np.zeros(0, int)

def evaluate_images(images_bgr, paths, conf=0.25, nms_iou=0.7, thr=0.5, model_=None, label_dir=None):
    model_ = model_ or best; tp = fp = fn = 0; tp_ious, best_ious, per_img = [], [], []
    for ip, img, r in zip(paths, images_bgr, model_.predict(images_bgr, conf=conf, iou=nms_iou, imgsz=IMGSZ, device=DEVICE, verbose=False, stream=True, batch=16)):
        g, gc = gt_boxes(ip, img.shape[:2], label_dir); pb = r.boxes.xyxy.cpu().numpy(); pc = r.boxes.cls.cpu().numpy().astype(int); ps = r.boxes.conf.cpu().numpy()
        iou = box_iou(torch.tensor(pb, dtype=torch.float32), torch.tensor(g, dtype=torch.float32)).numpy() if len(pb) and len(g) else np.zeros((len(pb), len(g)))
        used = set(); itp = 0
        for i in np.argsort(-ps):
            cand = [(iou[i, j], j) for j in range(len(g)) if j not in used and gc[j] == pc[i] and iou[i, j] >= thr]
            if cand: v, j = max(cand); used.add(j); itp += 1; tp_ious.append(v)
        ifp, ifn = len(pb)-itp, len(g)-itp; tp += itp; fp += ifp; fn += ifn
        for j in range(len(g)):
            same = [iou[i, j] for i in range(len(pb)) if pc[i] == gc[j]]; best_ious.append(max(same) if same else 0.0)
        per_img.append({"path": str(ip), "tp": itp, "fp": ifp, "fn": ifn, "n_gt": len(g)})
    P = tp/max(1, tp+fp); R = tp/max(1, tp+fn)
    return {"TP": tp, "FP": fp, "FN": fn, "precision": P, "recall": R, "F1": 2*P*R/max(1e-9, P+R), "mean_IoU_TP": float(np.mean(tp_ious)) if tp_ious else 0.0,
            "mean_best_IoU_per_GT": float(np.mean(best_ious)) if best_ious else 0.0, "tp_ious": tp_ious}, pd.DataFrame(per_img)

test_arrays = [cv2.imread(str(p)) for p in test_imgs]
IOU_RES, per_img_df = evaluate_images(test_arrays, test_imgs)
print({k: (round(v, 4) if isinstance(v, float) else v) for k, v in IOU_RES.items() if k != "tp_ious"})
plt.figure(figsize=(8, 4)); plt.hist(IOU_RES["tp_ious"], bins=30, color="tab:green", edgecolor="k"); plt.axvline(IOU_RES["mean_IoU_TP"], color="r", ls="--", label=f"mean = {IOU_RES['mean_IoU_TP']:.3f}")
plt.xlabel("IoU of matched (true-positive) detections"); plt.ylabel("count"); plt.title("IoU distribution on the test set"); plt.legend(); plt.tight_layout(); plt.savefig(FIG/"08_iou_hist.png", dpi=120); plt.show()

### 4.3 Inference speed
Latency is measured end-to-end for `model.predict()` on one image (pre-processing + network forward + NMS), 20 warm-up + 100 timed runs, plus batch throughput. The validator's own split-up (pre / inference / post, ms per image) is reported too.

In [ ]:
def bench(model_, img, n=100, warm=20, **kw):
    for _ in range(warm): model_.predict(img, verbose=False, **kw)
    ts = []
    for _ in range(n):
        if torch.cuda.is_available(): torch.cuda.synchronize()
        t = time.perf_counter(); model_.predict(img, verbose=False, **kw)
        if torch.cuda.is_available(): torch.cuda.synchronize()
        ts.append((time.perf_counter()-t)*1000)
    ts = np.array(ts); return {"mean_ms": ts.mean(), "std_ms": ts.std(), "p95_ms": np.percentile(ts, 95), "FPS": 1000/ts.mean()}

img0 = test_arrays[0]; speed_rows = []
for sz in (320, 640, 960):
    for half in ([False, True] if torch.cuda.is_available() else [False]):
        speed_rows.append({"device": GPU_NAME, "imgsz": sz, "precision": "FP16" if half else "FP32", "batch": 1, **bench(YOLO(str(BEST)), img0, imgsz=sz, half=half, device=DEVICE)})   # fresh model per config: the predictor caches half/device
batch16 = test_arrays[:16]; tb = []
_bm = YOLO(str(BEST))
for _ in range(10):
    t = time.perf_counter(); _bm.predict(batch16, imgsz=IMGSZ, half=torch.cuda.is_available(), device=DEVICE, verbose=False, batch=16); tb.append(time.perf_counter()-t)
tbm = float(np.mean(tb[2:])); speed_rows.append({"device": GPU_NAME, "imgsz": IMGSZ, "precision": "FP16" if torch.cuda.is_available() else "FP32", "batch": 16, "mean_ms": tbm*1000/16, "std_ms": float(np.std(tb[2:]))*1000/16, "p95_ms": np.nan, "FPS": 16/tbm})
cpu = bench(YOLO(str(BEST)), img0, n=15, warm=3, imgsz=IMGSZ, device="cpu"); speed_rows.append({"device": "CPU", "imgsz": IMGSZ, "precision": "FP32", "batch": 1, **cpu})
speed_df = pd.DataFrame(speed_rows); display(speed_df.round(2)); speed_df.to_csv(OUT/"inference_speed.csv", index=False)
print("Validator timing on test split (ms/img):", {k: round(v, 2) for k, v in S_test["speed_ms"].items()})

## 5. Testing on unseen data
The **test split** (never used for training or checkpoint selection) is used for: quantitative metrics (above), qualitative prediction grids, an inspection of the hardest images, and a **robustness study** with synthetic corruptions (darkening, blur, gaussian noise, JPEG compression) that mimic real-world acquisition shifts.

In [ ]:
def plot_preds(paths, conf=0.35, ncols=4, title="", fname=None, figsize=None):
    n = len(paths); nrows = math.ceil(n/ncols); fig, axs = plt.subplots(nrows, ncols, figsize=figsize or (5*ncols, 4*nrows)); axs = np.array(axs).reshape(-1)
    for a, r in zip(axs, best.predict([str(p) for p in paths], conf=conf, imgsz=IMGSZ, device=DEVICE, verbose=False, stream=True)):
        a.imshow(r.plot(line_width=3)[..., ::-1]); a.set_title(f"{Path(r.path).name} - {len(r.boxes)} det.", fontsize=9); a.axis("off")
    for a in axs[n:]: a.axis("off")
    plt.suptitle(title); plt.tight_layout()
    if fname: plt.savefig(FIG/fname, dpi=110)
    plt.show()

rng = random.Random(7); show = rng.sample(test_imgs, 12)
plot_preds(show, title="Predictions on UNSEEN test images (conf >= 0.35)", fname="09_test_predictions.png")

In [ ]:
# ---- Ground truth vs prediction, side by side ----
pick = rng.sample(test_imgs, 4)
fig, axs = plt.subplots(4, 2, figsize=(13, 20))
for k, ip in enumerate(pick):
    im = cv2.imread(str(ip)); lab, _ = parse_label_file(split_dirs["test"]/"labels"/f"{ip.stem}.txt")
    axs[k,0].imshow(draw_gt(im, lab)[..., ::-1]); axs[k,0].set_title(f"Ground truth - {ip.name}"); axs[k,0].axis("off")
    r = best.predict(im, conf=0.35, imgsz=IMGSZ, device=DEVICE, verbose=False)[0]; axs[k,1].imshow(r.plot(line_width=3)[..., ::-1]); axs[k,1].set_title("Prediction"); axs[k,1].axis("off")
plt.tight_layout(); plt.savefig(FIG/"10_gt_vs_pred.png", dpi=100); plt.show()

# ---- hardest test images (most FP+FN) ----
per_img_df["errors"] = per_img_df.fp + per_img_df.fn
hard = per_img_df.sort_values("errors", ascending=False).head(8); display(hard.assign(path=hard.path.map(lambda p: Path(p).name)))
plot_preds([Path(p) for p in hard.path.head(4)], conf=0.25, ncols=2, title="Hardest test images (most FP+FN) - failure analysis", fname="11_failure_cases.png", figsize=(14, 9))

In [ ]:
# ---- Robustness to corruptions on the unseen test set ----
def darken(im):  return np.clip(im.astype(np.float32)*0.45, 0, 255).astype(np.uint8)
def blur(im):    return cv2.GaussianBlur(im, (0, 0), sigmaX=4)
def noise(im):   return np.clip(im.astype(np.float32) + np.random.RandomState(0).normal(0, 25, im.shape), 0, 255).astype(np.uint8)
def jpeg(im):    return cv2.imdecode(cv2.imencode(".jpg", im, [cv2.IMWRITE_JPEG_QUALITY, 10])[1], 1)
def flip(im):    return im[:, ::-1].copy()
rob_rows = [{"condition": "clean", **{k: v for k, v in IOU_RES.items() if k != "tp_ious"}}]
for nm, fn_ in [("darker (x0.45)", darken), ("gaussian blur (sigma=4)", blur), ("noise (sd=25)", noise), ("JPEG q=10", jpeg)]:
    res, _ = evaluate_images([fn_(i) for i in test_arrays], test_imgs); rob_rows.append({"condition": nm, **{k: v for k, v in res.items() if k != "tp_ious"}})
rob_df = pd.DataFrame(rob_rows); display(rob_df.round(4)); rob_df.to_csv(OUT/"robustness.csv", index=False)
fig, ax = plt.subplots(figsize=(10, 4)); rob_df.set_index("condition")[["precision", "recall", "F1", "mean_IoU_TP"]].plot.bar(ax=ax, rot=15); ax.set_ylim(0, 1.05); ax.set_title("Robustness of the detector on corrupted test images (conf 0.25, IoU 0.5)")
plt.tight_layout(); plt.savefig(FIG/"12_robustness.png", dpi=120); plt.show()
cond_imgs = [("clean", test_arrays[3]), ("darker", darken(test_arrays[3])), ("blur", blur(test_arrays[3])), ("noise", noise(test_arrays[3])), ("JPEG q10", jpeg(test_arrays[3]))]
fig, axs = plt.subplots(1, 5, figsize=(22, 4.5))
for a, (nm, im) in zip(axs, cond_imgs): a.imshow(best.predict(im, conf=0.25, imgsz=IMGSZ, device=DEVICE, verbose=False)[0].plot()[..., ::-1]); a.set_title(nm); a.axis("off")
plt.tight_layout(); plt.savefig(FIG/"13_robustness_examples.png", dpi=110); plt.show()

## 5b. Custom annotated images (our own data)
In addition to the dataset's test split, **6 custom photographs** were collected from Wikimedia Commons (free CC licences; sources listed below), **manually annotated** with bounding boxes (`x1, y1, x2, y2` in pixels of the images embedded here) and converted to YOLO format. They come from a different source than the training data, so they are a genuinely independent check. They are used for **evaluation only** (never for training).

In [ ]:
import base64, io, zipfile
CUSTOM_ZIP_B64 = "..."
CUSTOM = WORK / "custom"; (CUSTOM/"images").mkdir(parents=True, exist_ok=True); (CUSTOM/"labels").mkdir(exist_ok=True)
zipfile.ZipFile(io.BytesIO(base64.b64decode(CUSTOM_ZIP_B64))).extractall(CUSTOM/"images")

# ---- MANUAL ANNOTATIONS: one entry per object, pixel box (x1, y1, x2, y2) ----
ANNOTATIONS = [
    {"file": "buffalo1.jpg", "class": "buffalo", "box": [171, 125, 608, 473]},
    {"file": "elephant1.jpg", "class": "elephant", "box": [279, 73, 568, 500]},
    {"file": "elephant1.jpg", "class": "elephant", "box": [558, 308, 667, 468]},
    {"file": "rhino1.jpg", "class": "rhino", "box": [102, 92, 457, 457]},
    {"file": "rhino1.jpg", "class": "rhino", "box": [530, 173, 695, 439]},
    {"file": "rhino2.jpg", "class": "rhino", "box": [125, 43, 656, 535]},
    {"file": "zebra1.jpg", "class": "zebra", "box": [92, 332, 262, 430]},
    {"file": "zebra1.jpg", "class": "zebra", "box": [288, 328, 428, 426]},
    {"file": "zebra2.jpg", "class": "zebra", "box": [116, 176, 516, 457]}
]
# ---- image sources / licences ----
CUSTOM_SOURCES = [
    {"file": "buffalo1.jpg", "title": "File:African Buffalo.JPG", "license": "CC BY 2.5", "site": "commons.wikimedia.org"},
    {"file": "elephant1.jpg", "title": "File:African bush elephants (Loxodonta africana) female with six-week-old baby.jpg", "license": "CC BY-SA 4.0", "site": "commons.wikimedia.org"},
    {"file": "rhino1.jpg", "title": "Open Images (via Kaggle antoreepjana/animals-detection-images-dataset) test/Rhinoceros/561c4dccbbfb86f1.jpg", "license": "CC BY 2.0 (Open Images)", "site": "kaggle.com"},
    {"file": "rhino2.jpg", "title": "Open Images (via Kaggle antoreepjana/animals-detection-images-dataset) test/Rhinoceros/2e3a7f4b6057b388.jpg", "license": "CC BY 2.0 (Open Images)", "site": "kaggle.com"},
    {"file": "zebra1.jpg", "title": "Open Images (via Kaggle antoreepjana/animals-detection-images-dataset) test/Zebra/099f5af7592c9fcc.jpg", "license": "CC BY 2.0 (Open Images)", "site": "kaggle.com"},
    {"file": "zebra2.jpg", "title": "Open Images (via Kaggle antoreepjana/animals-detection-images-dataset) test/Zebra/02e850bb904d9467.jpg", "license": "CC BY 2.0 (Open Images)", "site": "kaggle.com"}
]

In [ ]:
# ---- convert the manual pixel boxes to YOLO txt labels (class xc yc w h, normalised) and visualise ----
ann_df = pd.DataFrame([{"file": a["file"], "class": a["class"], "x1": a["box"][0], "y1": a["box"][1], "x2": a["box"][2], "y2": a["box"][3]} for a in ANNOTATIONS])
for f, g in ann_df.groupby("file"):
    W_, H_ = Image.open(CUSTOM/"images"/f).size; lines = []
    for _, r_ in g.iterrows():
        assert 0 <= r_.x1 < r_.x2 <= W_ and 0 <= r_.y1 < r_.y2 <= H_, (f, r_.to_dict())
        lines.append(f"{NAMES.index(r_['class'])} {(r_.x1+r_.x2)/2/W_:.6f} {(r_.y1+r_.y2)/2/H_:.6f} {(r_.x2-r_.x1)/W_:.6f} {(r_.y2-r_.y1)/H_:.6f}")
    (CUSTOM/"labels"/f"{Path(f).stem}.txt").write_text("\n".join(lines) + "\n")
ann_df.to_csv(OUT/"custom_annotations.csv", index=False); display(ann_df)
print("YOLO label example:", (CUSTOM/"labels"/f"{Path(ann_df.file[0]).stem}.txt").read_text().strip())
custom_imgs = sorted((CUSTOM/"images").glob("*.jpg"))
nc_ = len(custom_imgs); ncol = min(3, nc_); fig, axs = plt.subplots(math.ceil(nc_/ncol), ncol, figsize=(6*ncol, 4.6*math.ceil(nc_/ncol))); axs = np.array(axs).reshape(-1)
for a, ip in zip(axs, custom_imgs):
    lab, _ = parse_label_file(CUSTOM/"labels"/f"{ip.stem}.txt"); a.imshow(draw_gt(cv2.imread(str(ip)), lab)[..., ::-1]); a.set_title(f"{ip.name} ({len(lab)} annotated objects)"); a.axis("off")
for a in axs[nc_:]: a.axis("off")
plt.suptitle("Custom images with our manual annotations"); plt.tight_layout(); plt.savefig(FIG/"16_custom_annotations.png", dpi=110); plt.show()

In [ ]:
# ---- evaluate the trained model on the custom annotated images ----
CUSTOM_YAML = CUSTOM/"custom.yaml"
yaml.safe_dump({"path": str(CUSTOM), "train": "images", "val": "images", "test": "images", "nc": NC, "names": {i: n for i, n in enumerate(NAMES)}}, open(CUSTOM_YAML, "w"))
mc = best.val(data=str(CUSTOM_YAML), split="val", imgsz=IMGSZ, batch=8, conf=0.001, iou=0.6, device=DEVICE, plots=False, project=str(RUNS), name="eval_custom", exist_ok=True, verbose=False)
CUSTOM_ARR = [cv2.imread(str(p)) for p in custom_imgs]
CUSTOM_IOU, custom_img_df = evaluate_images(CUSTOM_ARR, custom_imgs, label_dir=CUSTOM/"labels")
CUSTOM_RES = {"n_images": len(custom_imgs), "n_objects": int(len(ann_df)), "precision": float(mc.box.mp), "recall": float(mc.box.mr), "mAP@0.5": float(mc.box.map50), "mAP@0.5:0.95": float(mc.box.map),
              **{"@conf0.25_" + k: v for k, v in CUSTOM_IOU.items() if k != "tp_ious"}}
print({k: (round(v, 4) if isinstance(v, float) else v) for k, v in CUSTOM_RES.items()})
custom_img_df["image"] = custom_img_df.path.map(lambda p: Path(p).name); display(custom_img_df[["image", "n_gt", "tp", "fp", "fn"]]); custom_img_df.to_csv(OUT/"custom_per_image.csv", index=False)
fig, axs = plt.subplots(len(custom_imgs), 2, figsize=(13, 4.6*len(custom_imgs)))
for k, (ip, im) in enumerate(zip(custom_imgs, CUSTOM_ARR)):
    lab, _ = parse_label_file(CUSTOM/"labels"/f"{ip.stem}.txt"); axs[k, 0].imshow(draw_gt(im, lab)[..., ::-1]); axs[k, 0].set_title(f"Our annotation - {ip.name}"); axs[k, 0].axis("off")
    r = best.predict(im, conf=0.25, imgsz=IMGSZ, device=DEVICE, verbose=False)[0]; axs[k, 1].imshow(r.plot(line_width=3)[..., ::-1]); axs[k, 1].set_title(f"YOLO11 prediction ({len(r.boxes)} detections)"); axs[k, 1].axis("off")
plt.tight_layout(); plt.savefig(FIG/"17_custom_gt_vs_pred.png", dpi=100); plt.show()

## 6. Live prediction on frames
A real-time inference loop that reads a video **frame by frame** (exactly the loop used for a webcam / RTSP stream – see `live_predict.py` for the standalone script) and runs YOLO11 on every frame, overlaying detections, per-frame latency and running FPS.
The Kaggle notebook has no camera, so a demonstration video is synthesised from **unseen test images** (each shown as a short clip with a slow camera zoom, producing a continuous frame stream with moving content).

In [ ]:
VID = OUT/"videos"; VID.mkdir(exist_ok=True)
W, H, FPS_IN = 1280, 720, 30
n_clip, hold = (4, 20) if SMOKE else (30, 24)
clip_imgs = random.Random(11).sample(test_imgs, n_clip)
src_path = VID/"demo_input.mp4"
vw = cv2.VideoWriter(str(src_path), cv2.VideoWriter_fourcc(*"mp4v"), FPS_IN, (W, H))
for ip in clip_imgs:
    im = cv2.imread(str(ip)); h, w = im.shape[:2]; s = max(W/w, H/h); im = cv2.resize(im, (int(w*s)+1, int(h*s)+1)); h, w = im.shape[:2]
    for f in range(hold):
        z = 1.0 + 0.12*f/hold; cw, ch = int(W/z), int(H/z); x0, y0 = (w-cw)//2, (h-ch)//2     # slow centre zoom
        vw.write(cv2.resize(im[y0:y0+ch, x0:x0+cw], (W, H)))
vw.release()
print("input video:", src_path, f"{n_clip*hold} frames @ {FPS_IN} fps")

In [ ]:
# ---- Live frame-by-frame prediction loop ----
LIVE_CONF = 0.35
cap = cv2.VideoCapture(str(src_path)); fps_in = cap.get(cv2.CAP_PROP_FPS)
out_path = VID/"live_prediction_raw.mp4"; writer = cv2.VideoWriter(str(out_path), cv2.VideoWriter_fourcc(*"mp4v"), fps_in, (W, H))
live_model = YOLO(str(BEST))
for _ in range(5): live_model.predict(np.zeros((H, W, 3), np.uint8), imgsz=IMGSZ, half=torch.cuda.is_available(), device=DEVICE, verbose=False)      # warm-up
lat, ndet, grabbed_frames, t_start, idx = [], [], {}, time.perf_counter(), 0
while True:
    ok, frame = cap.read()
    if not ok: break
    t = time.perf_counter()
    r = live_model.predict(frame, conf=LIVE_CONF, imgsz=IMGSZ, half=torch.cuda.is_available(), device=DEVICE, verbose=False)[0]
    dt = (time.perf_counter()-t)*1000; lat.append(dt); ndet.append(len(r.boxes))
    vis = r.plot(line_width=3)
    run_fps = (idx+1)/(time.perf_counter()-t_start)
    cv2.rectangle(vis, (0, 0), (520, 44), (0, 0, 0), -1); cv2.putText(vis, f"YOLO11s | {dt:5.1f} ms | {run_fps:5.1f} FPS | {len(r.boxes)} obj", (8, 30), cv2.FONT_HERSHEY_SIMPLEX, .8, (0, 255, 0), 2)
    writer.write(vis)
    if idx % max(1, (hold*n_clip)//8) == hold//2: grabbed_frames[idx] = vis[..., ::-1].copy()
    idx += 1
cap.release(); writer.release()
LIVE = {"frames": idx, "avg_latency_ms": float(np.mean(lat)), "p95_latency_ms": float(np.percentile(lat, 95)), "avg_FPS_end_to_end": idx/(time.perf_counter()-t_start), "mean_detections_per_frame": float(np.mean(ndet))}
print(json.dumps({k: round(v, 2) for k, v in LIVE.items()}, indent=1))
fig, ax = plt.subplots(1, 2, figsize=(15, 3.8)); ax[0].plot(lat); ax[0].set_xlabel("frame"); ax[0].set_ylabel("latency (ms)"); ax[0].set_title("Per-frame latency (live loop)"); ax[1].plot(ndet, color="tab:orange"); ax[1].set_xlabel("frame"); ax[1].set_ylabel("# detections"); ax[1].set_title("Detections per frame")
plt.tight_layout(); plt.savefig(FIG/"14_live_latency.png", dpi=120); plt.show()
ks = sorted(grabbed_frames)[:8]; fig, axs = plt.subplots(2, 4, figsize=(20, 6))
for a, k in zip(axs.ravel(), ks): a.imshow(grabbed_frames[k]); a.set_title(f"frame {k}"); a.axis("off")
plt.suptitle("Live prediction - sampled frames"); plt.tight_layout(); plt.savefig(FIG/"15_live_frames.png", dpi=100); plt.show()

In [ ]:
# ---- Multi-object tracking on the stream (persistent IDs, ByteTrack) + browser-friendly encodes + GIF ----
TRACK_OK = False
try:
    track_model = YOLO(str(BEST)); cap = cv2.VideoCapture(str(src_path)); tw = cv2.VideoWriter(str(VID/"live_tracking_raw.mp4"), cv2.VideoWriter_fourcc(*"mp4v"), fps_in, (W, H)); ids = set(); tl = []
    while True:
        ok, frame = cap.read()
        if not ok: break
        t = time.perf_counter(); r = track_model.track(frame, persist=True, tracker="bytetrack.yaml", conf=LIVE_CONF, imgsz=IMGSZ, device=DEVICE, verbose=False)[0]; tl.append((time.perf_counter()-t)*1000)
        if r.boxes.id is not None: ids.update(r.boxes.id.int().tolist())
        tw.write(r.plot(line_width=3))
    cap.release(); tw.release(); TRACK_OK = True
    print(f"tracking done: {len(ids)} unique track IDs, {np.mean(tl):.1f} ms/frame ({1000/np.mean(tl):.1f} FPS)")
except Exception as e: print("tracking skipped:", repr(e)[:300])

ff = shutil.which("ffmpeg")
for raw, fin in [("live_prediction_raw.mp4", "live_prediction.mp4"), ("live_tracking_raw.mp4", "live_tracking.mp4"), ("demo_input.mp4", "demo_input_h264.mp4")]:
    rp = VID/raw
    if not rp.exists(): continue
    if ff: subprocess.run([ff, "-y", "-loglevel", "error", "-i", str(rp), "-vcodec", "libx264", "-pix_fmt", "yuv420p", "-crf", "24", str(VID/fin)])
    else: shutil.copy(rp, VID/fin)
    if (VID/fin).exists(): rp.unlink()
# GIF preview (every 6th frame, 480px wide)
cap = cv2.VideoCapture(str(VID/"live_prediction.mp4")); gif, i = [], 0
while True:
    ok, fr = cap.read()
    if not ok: break
    if i % 6 == 0: gif.append(Image.fromarray(cv2.resize(fr, (480, 270))[..., ::-1]))
    i += 1
cap.release(); gif[0].save(VID/"live_prediction_preview.gif", save_all=True, append_images=gif[1:], duration=200, loop=0, optimize=True)
print({p.name: f"{p.stat().st_size/1e6:.1f} MB" for p in sorted(VID.iterdir())})
display(IPyImage(filename=str(VID/"live_prediction_preview.gif")))

## 7. Export, summary & deliverables

In [ ]:
# ---- Export to ONNX (deployment) and sanity-check it ----
ONNX_INFO = {}
try:
    onnx_path = best.export(format="onnx", imgsz=IMGSZ, simplify=True, half=False, device=DEVICE if DEVICE != "cpu" else "cpu")
    ONNX_INFO = {"path": str(onnx_path), "size_MB": round(Path(onnx_path).stat().st_size/1e6, 1)}
    r_onnx = YOLO(str(onnx_path), task="detect").predict(test_arrays[0], imgsz=IMGSZ, conf=0.35, verbose=False)[0]; ONNX_INFO["detections_on_sample"] = len(r_onnx.boxes)
    r_pt = best.predict(test_arrays[0], imgsz=IMGSZ, conf=0.35, device=DEVICE, verbose=False)[0]; ONNX_INFO["detections_pt_same_image"] = len(r_pt.boxes)
except Exception as e: ONNX_INFO = {"error": repr(e)[:300]}
print(ONNX_INFO)

In [ ]:
# ---- Collect deliverables ----
DEL = WORK / "deliverables"; (DEL/"figures").mkdir(parents=True, exist_ok=True); (DEL/"ultralytics_plots").mkdir(exist_ok=True); (DEL/"videos").mkdir(exist_ok=True)
shutil.copy(BEST, DEL/"yolo11s_wildlife_best.pt"); shutil.copy(SAVE/"weights"/"last.pt", DEL/"yolo11s_wildlife_last.pt")
if ONNX_INFO.get("path"): shutil.copy(ONNX_INFO["path"], DEL/"yolo11s_wildlife_best.onnx")
for f in FIG.glob("*.png"): shutil.copy(f, DEL/"figures"/f.name)
for f in list(OUT.glob("*.csv")): shutil.copy(f, DEL/f.name)
for f in SAVE.glob("*.png"): shutil.copy(f, DEL/"ultralytics_plots"/f"train_{f.name}")
for f in SAVE.glob("*.jpg"): shutil.copy(f, DEL/"ultralytics_plots"/f"train_{f.name}")
shutil.copy(SAVE/"results.csv", DEL/"ultralytics_results.csv"); shutil.copy(SAVE/"args.yaml", DEL/"train_args.yaml")
for sp in ("val", "test"):
    for f in Path({"val": m_val, "test": m_test}[sp].save_dir).glob("*.png"): shutil.copy(f, DEL/"ultralytics_plots"/f"eval_{sp}_{f.name}")
for f in VID.iterdir(): shutil.copy(f, DEL/"videos"/f.name)
shutil.copy(DATA_YAML, DEL/"wildlife_data.yaml")
if globals().get("CUSTOM"): shutil.copytree(CUSTOM, DEL/"custom_annotated_images", dirs_exist_ok=True)

best_row = hist.loc[hist["metrics/mAP50-95(B)"].idxmax()]
RESULTS = {"task": "Object detection with YOLOv11", "model": "yolo11s (COCO-pretrained, fine-tuned)", "dataset": "Wildlife YOLO-format (buffalo, elephant, rhino, zebra) - Kaggle ankanghosh651",
           "classes": NAMES, "ultralytics": ultralytics.__version__, "torch": torch.__version__, "gpu": GPU_NAME, "smoke_run": SMOKE,
           "params_M": round(n_params/1e6, 2), "GFLOPs": round(gflops, 1), "layers": n_layers,
           "train": {"epochs_requested": EPOCHS, "epochs_run": int(len(hist)), "best_epoch": int(best_row["epoch"]), "imgsz": IMGSZ, "batch": BATCH, "train_minutes": round(TRAIN_MIN, 1)},
           "dataset_audit": audit, "val": {k: v for k, v in S_val.items()}, "test": {k: v for k, v in S_test.items()},
           "per_class_test": PC_test.round(4).to_dict("records"), "per_class_val": PC_val.round(4).to_dict("records"),
           "iou_analysis_test": {k: v for k, v in IOU_RES.items() if k != "tp_ious"}, "inference_speed": speed_df.round(3).replace({np.nan: None}).to_dict("records"),
           "robustness": rob_df.round(4).to_dict("records"), "custom_images": globals().get("CUSTOM_RES"), "live_prediction": LIVE, "onnx": ONNX_INFO}
json.dump(RESULTS, open(DEL/"results_summary.json", "w"), indent=2, default=float)
ZIP_PATH = shutil.make_archive(str(WORK / "yolo11_deliverables"), "zip", DEL)
shutil.rmtree(DATA, ignore_errors=True)                       # remove the copied dataset from the output
for p in RUNS.glob("*"): 
    if p.is_dir() and p.name.startswith("eval_"): shutil.rmtree(p, ignore_errors=True)
print(json.dumps({k: RESULTS[k] for k in ["model", "params_M", "GFLOPs", "train"]}, indent=1))
try:
    from google.colab import files as _cf; _cf.download(ZIP_PATH)          # Colab: download the deliverables zip
except Exception: pass
print("TEST  :", {k: round(v, 4) for k, v in S_test.items() if k not in ("split", "speed_ms")})
print("IoU   :", {k: round(v, 4) if isinstance(v, float) else v for k, v in IOU_RES.items() if k != "tp_ious"})
print("LIVE  :", {k: round(v, 2) for k, v in LIVE.items()})
for p in sorted(DEL.rglob("*")):
    if p.is_file(): print(f"  {str(p.relative_to(DEL)):60s} {p.stat().st_size/1e6:8.2f} MB")

## 8. Conclusion
See the printed metrics above and `results_summary.json`; the accompanying report (`REPORT.md` / `.docx`) interprets them. Key take-aways: transfer learning from COCO lets `yolo11s` reach high mAP on a small custom dataset within one Kaggle GPU session; the held-out test split confirms generalisation; the model runs far faster than real-time (tens to hundreds of FPS on a T4) so it is suitable for live frame prediction.